# 크라베오 AI 재학습 — v2
# Cravveo AI Retrain — v2

**변경 사항 | Changes:**
- 기존 데이터 83개 + Obsidian 데이터 203개 = 총 286개
- Existing 83 + Obsidian 203 = Total 286 samples

**순서 | Steps:**
1. 설치
2. 모델 로드
3. LoRA 어댑터
4. 데이터셋 로드 + 합치기
5. 학습
6. 저장 + HuggingFace 업로드
7. GGUF 변환

In [ ]:
# 셀 1: 필수 라이브러리 설치
# Cell 1: Install required libraries
!pip install unsloth trl datasets

In [ ]:
# 셀 2: CUDA 오류 방지 + 모델 로드
# Cell 2: CUDA fix + model load
import ctypes
ctypes.CDLL("libnvJitLink.so.13", mode=ctypes.RTLD_GLOBAL)

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=512,
    load_in_4bit=True,
)
print("모델 로드 완료 | Model loaded")

In [ ]:
# 셀 3: LoRA 어댑터 설정
# Cell 3: LoRA adapter setup
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("LoRA 어댑터 적용 완료 | LoRA adapter applied")

In [ ]:
# 셀 4: 데이터셋 로드 + 합치기
# Cell 4: Load datasets + combine
from datasets import load_dataset, Dataset, concatenate_datasets
import json

# --- 기존 데이터셋 (HuggingFace) ---
# instruction/output 형식 → 질문:/답변: 형식으로 변환
existing_raw = load_dataset("cravveo/cravveo-briefing-dataset", split="train")

def convert_existing(example):
    return {"text": f"질문: {example['instruction']}\n답변: {example['output']}"}

existing = existing_raw.map(convert_existing, remove_columns=existing_raw.column_names)
print(f"기존 데이터셋: {len(existing)}개")

# --- 새 Obsidian 데이터셋 (파일 업로드) ---
from google.colab import files
print("obsidian_dataset.jsonl 파일을 업로드하세요 | Upload obsidian_dataset.jsonl")
uploaded = files.upload()

new_data = []
with open("obsidian_dataset.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        new_data.append(json.loads(line.strip()))

new_dataset = Dataset.from_list(new_data)
print(f"Obsidian 데이터셋: {len(new_dataset)}개")

# --- 합치기 ---
combined = concatenate_datasets([existing, new_dataset])
print(f"\n합계 | Total: {len(combined)}개")
print("샘플 확인 | Sample check:")
print(combined[0]['text'])
print("---")
print(combined[-1]['text'])

In [ ]:
# 셀 5: 학습
# Cell 5: Training
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=combined,
    dataset_text_field="text",
    max_seq_length=512,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        output_dir="cravveo_v2_output",
        save_strategy="no",
    ),
)

print("학습 시작 | Training start...")
trainer.train()
print("학습 완료 | Training complete!")

In [ ]:
# 셀 6: 테스트 — 학습 결과 확인
# Cell 6: Quick test
FastLanguageModel.for_inference(model)

inputs = tokenizer(["질문: 크라베오 컴퍼니가 뭐야?\n답변: "], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.2)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# 셀 7: HuggingFace 업로드
# Cell 7: Push to HuggingFace
from google.colab import userdata
from huggingface_hub import login

# Colab Secrets에서 HF_TOKEN 읽기
# Read HF_TOKEN from Colab Secrets
token = userdata.get('HF_TOKEN')
login(token=token)

model.save_pretrained("cravveo-llama-lora-v2")
model.push_to_hub("cravveo/cravveo-llama-lora", token=token)  # 같은 레포에 덮어쓰기
tokenizer.push_to_hub("cravveo/cravveo-llama-lora", token=token)

print("HuggingFace 업로드 완료 | HuggingFace upload complete")
print("https://huggingface.co/cravveo/cravveo-llama-lora")

In [ ]:
# 셀 8: GGUF 변환 (q8_0)
# Cell 8: GGUF conversion
model.save_pretrained_gguf(
    "cravveo-v2-gguf",
    tokenizer,
    quantization_method="q8_0"
)

# 파일 다운로드
# Download the file
from google.colab import files
import os

gguf_file = [f for f in os.listdir("cravveo-v2-gguf") if f.endswith(".gguf")][0]
print(f"GGUF 파일: {gguf_file}")
files.download(f"cravveo-v2-gguf/{gguf_file}")
print("다운로드 완료 | Download complete")